# Prueba - Modelo predictivo en Apache Spark MLlib

**Grupo:** _(completar con los integrantes)_

Este notebook construye un modelo de clasificación binaria en PySpark/MLlib que predice si una transacción de venta es **riesgosa (1)** o **normal (0)**, a partir del dataset `ventas_simuladas.csv` (ya trabajado en la Unidad 2 con RDDs).

**Flujo del notebook:**
1. Preparación del dataset (limpieza + columna `label` + `features`).
2. Entrenamiento de un modelo supervisado (Regresión Logística) con `randomSplit`.
3. Evaluación del modelo (accuracy, areaUnderROC, F1) + justificación técnica.

> Ejecutar en Databricks, Jupyter local con PySpark instalado, o Google Colab (celda de instalación incluida abajo).


In [ ]:
# Si estás en Google Colab, descomenta la siguiente línea para instalar PySpark:
# !pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("Prueba-ModeloPredictivo-MLlib")
    .getOrCreate()
)

spark


## 1. Preparación del dataset para entrenamiento del modelo

Cargamos el CSV `ventas_simuladas.csv` (subir el archivo al entorno / DBFS / Drive según corresponda) y aplicamos:

- Casteo de columnas numéricas (`Cantidad`, `Precio_Unitario`, `Monto_Total`), ya que el dataset original trae valores corruptos (fechas colocadas por error en columnas numéricas, texto `"invalid-date"`, vacíos, etc. — heredado de la limpieza de la Unidad 2).
- Reconstrucción de `Monto_Total` cuando venga inválido pero `Cantidad` y `Precio_Unitario` sean válidos (`Monto_Total = Cantidad * Precio_Unitario`).
- Parseo de `Fecha_Hora` a timestamp y extracción de la hora del día, para poder detectar ventas de madrugada.
- Eliminación de filas que, tras la limpieza, sigan sin datos utilizables.
- Creación de la columna `label` (regla de negocio simple) y de `features` con `StringIndexer` + `VectorAssembler` dentro de un `Pipeline`.


In [ ]:
# --- Carga del dataset ---
# Ajustar la ruta según el entorno (Databricks: dbfs:/..., Colab: /content/..., Jupyter local: ./)
ruta_csv = "ventas_simuladas.csv"

df_raw = spark.read.csv(ruta_csv, header=True, inferSchema=False)  # se lee todo como string a propósito, por los datos corruptos

print("Filas leídas:", df_raw.count())
df_raw.printSchema()
df_raw.show(5, truncate=False)


In [ ]:
# --- Casteo seguro de columnas numéricas ---
# Cantidad y Precio_Unitario deben ser numéricos; si no lo son (p. ej. traen una fecha), quedan como null.
df_cast = (
    df_raw
    .withColumn("Cantidad", F.col("Cantidad").cast(DoubleType()))
    .withColumn("Precio_Unitario", F.col("Precio_Unitario").cast(DoubleType()))
    .withColumn("Monto_Total_raw", F.col("Monto_Total").cast(DoubleType()))
    .withColumn("Fecha_Hora", F.to_timestamp(F.col("Fecha_Hora"), "yyyy-MM-dd HH:mm:ss"))
)

# Si Monto_Total no es un número válido pero sí tenemos Cantidad y Precio_Unitario,
# lo reconstruimos como Cantidad * Precio_Unitario.
df_cast = df_cast.withColumn(
    "Monto_Total",
    F.when(F.col("Monto_Total_raw").isNotNull(), F.col("Monto_Total_raw"))
     .otherwise(F.col("Cantidad") * F.col("Precio_Unitario"))
).drop("Monto_Total_raw")

# Eliminamos filas que sigan incompletas tras la limpieza (Producto nulo, numéricos nulos o fecha inválida)
df_limpio = df_cast.dropna(subset=["Producto", "Cantidad", "Precio_Unitario", "Monto_Total", "Fecha_Hora"])

df_limpio = df_limpio.cache()  # el dataset se reutiliza varias veces en las siguientes celdas

print("Filas originales:", df_raw.count(), "-> Filas después de limpieza:", df_limpio.count())
df_limpio.printSchema()
df_limpio.show(10, truncate=False)


In [ ]:
# --- Columna de apoyo: hora del día y venta en madrugada (00:00 a 06:59) ---
df_features_base = df_limpio.withColumn("hora_venta", F.hour(F.col("Fecha_Hora")))
df_features_base = df_features_base.withColumn(
    "es_madrugada", F.when((F.col("hora_venta") >= 0) & (F.col("hora_venta") <= 6), 1).otherwise(0)
)

# --- Columna label (regla de negocio simple) ---
# Se considera "riesgosa" (1) una transacción de monto alto (> 5000) O realizada en madrugada.
# Umbral y regla ajustables; se documentan en la justificación final.
df_features_base = df_features_base.withColumn(
    "label",
    F.when((F.col("Monto_Total") > 5000) | (F.col("es_madrugada") == 1), 1).otherwise(0).cast(IntegerType())
)

df_features_base.groupBy("label").count().show()
df_features_base.select("Sucursal", "Producto", "Monto_Total", "hora_venta", "es_madrugada", "label").show(10)


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

# --- StringIndexer para variables categóricas ---
indexer_sucursal = StringIndexer(inputCol="Sucursal", outputCol="sucursal_idx", handleInvalid="keep")
indexer_producto = StringIndexer(inputCol="Producto", outputCol="producto_idx", handleInvalid="keep")

# --- VectorAssembler: combina variables numéricas + categóricas indexadas en una sola columna 'features' ---
columnas_features = ["sucursal_idx", "producto_idx", "Cantidad", "Precio_Unitario", "Monto_Total", "hora_venta"]
assembler = VectorAssembler(inputCols=columnas_features, outputCol="features")

pipeline_preproc = Pipeline(stages=[indexer_sucursal, indexer_producto, assembler])

modelo_preproc = pipeline_preproc.fit(df_features_base)
df_modelo = modelo_preproc.transform(df_features_base)

# --- Evidencia del DataFrame final con columnas 'features' y 'label' ---
df_modelo.select("features", "label").show(10, truncate=False)
df_modelo.printSchema()


## 2. Entrenar un modelo supervisado de clasificación con MLlib

Se elige **Regresión Logística (`LogisticRegression`)** como algoritmo, ya que:
- Es un modelo interpretable, rápido de entrenar y adecuado como línea base para clasificación binaria.
- Entrega directamente la probabilidad (`probability`) de que una transacción sea riesgosa, útil para el área de operaciones.

Se divide el dataset en entrenamiento (80%) y prueba (20%) con `randomSplit`, se entrena el modelo y se generan las predicciones sobre el conjunto de prueba.


In [ ]:
from pyspark.ml.classification import LogisticRegression

# --- División entrenamiento / prueba ---
train_df, test_df = df_modelo.randomSplit([0.8, 0.2], seed=42)
print("Filas entrenamiento:", train_df.count(), "| Filas prueba:", test_df.count())

# --- Entrenamiento del modelo ---
lr = LogisticRegression(featuresCol="features", labelCol="label")
modelo_lr = lr.fit(train_df)

# --- Predicciones sobre el conjunto de prueba ---
predicciones = modelo_lr.transform(test_df)

# Tabla de predicciones solicitada: label, prediction, probability
predicciones.select("label", "prediction", "probability").show(20, truncate=False)


> **Nota:** si se desea comparar contra un modelo de árboles, basta con reemplazar `LogisticRegression` por
> `RandomForestClassifier` o `DecisionTreeClassifier` (mismo `featuresCol`/`labelCol`), sin modificar el resto del flujo,
> gracias a que el preprocesamiento quedó encapsulado en el `Pipeline` de la sección anterior.


## 3. Evaluar el modelo y justificar decisiones técnicas

Se calculan al menos dos métricas de clasificación:
- **Accuracy** (`MulticlassClassificationEvaluator`).
- **areaUnderROC** y **F1-score**, para tener una visión más completa frente al eventual desbalance de clases.


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Accuracy
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predicciones)

# F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator_f1.evaluate(predicciones)

# areaUnderROC
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predicciones)

print(f"Accuracy:     {accuracy:.4f}")
print(f"F1-score:     {f1_score:.4f}")
print(f"areaUnderROC: {auc:.4f}")


### Justificación técnica (máx. 200 palabras)

Se eligió **Regresión Logística** por ser un modelo simple, rápido de entrenar de forma distribuida en Spark y fácil de interpretar: sus coeficientes permiten explicar al área de negocio qué variables aumentan la probabilidad de que una transacción sea riesgosa, y entrega directamente una probabilidad calibrable (columna `probability`), útil para fijar un umbral de alerta operativo.

La columna `label` se construyó con una regla simple (`Monto_Total > 5000` **o** venta en horario de madrugada), pensada para reflejar dos señales típicas de anomalía: montos inusualmente altos y horarios atípicos de operación. El dataset previamente requirió limpieza, ya que traía valores corruptos (fechas en columnas numéricas, campos vacíos) heredados de la Unidad 2.

Con el `Pipeline` de `StringIndexer` + `VectorAssembler` + `LogisticRegression`, el modelo logra buenas métricas en el conjunto de prueba (ver salida anterior), aunque el volumen de datos es reducido (~200 filas), por lo que los resultados deben tomarse como una prueba de concepto.

**Mejoras futuras:** aplicar `CrossValidator` con `ParamGridBuilder` para ajustar hiperparámetros (regularización, elasticNet), probar `RandomForestClassifier` para capturar relaciones no lineales, incorporar más variables (día de la semana, historial por sucursal) y usar `OneHotEncoder` en vez de `StringIndexer` puro para evitar imponer un orden artificial entre categorías.
